# Backward Universal Lab (DEPRECATED)

**Deprecated.** Use **05_backward_wrought.ipynb** instead. That notebook uses the synthetic search pool from **06_generate_synthetic_wrought.ipynb** and per-target config; no on-the-fly GMM training.

This notebook is kept for reference only. It trains an ensemble and GMM on the fly and samples 50k compositions per run. The current architecture uses a pre-generated wrought synthetic database as the search pool.

In [1]:
import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.mixture import GaussianMixture
from sklearn.metrics import r2_score, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')

try:
    from utils import load_hyperparams, get_default_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, get_default_hyperparams

FILE_NAME = 'wrought_alloys_final.csv'


In [2]:
# MODULE 1: UNIVERSAL DATA INGESTION
class TitaniumDataProcessor:
    def __init__(self, file_path):
        self.file_path = file_path
        self.raw_data = None
        self.clean_data = None
        self.inputs = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti']
        self.targets = ['UTS (MPa)', 'Yield Strength (MPa)', 'Elec. Conductivity Vol (% IACS)']
        self.target_map = {
            'UTS (MPa)': ['UTS (MPa)', 'Tensile Strength: Ultimate (UTS) (MPa)'],
            'Yield Strength (MPa)': ['Yield Strength (MPa)', 'YS (MPa)', 'Tensile Strength: Yield (Proof) (MPa)'],
            'Elec. Conductivity Vol (% IACS)': ['Elec. Conductivity Vol (% IACS)', 'Electrical Conductivity: Equal Volume (% IACS)']
        }

    def load_and_process(self):
        print(f"[{'='*25} PHASE 1: DEEP DATA MINING {'='*25}")
        try:
            if self.file_path.endswith('.xlsx') or self.file_path.endswith('.xls'):
                self.raw_data = pd.read_excel(self.file_path)
            else:
                with open(self.file_path, 'rb') as f:
                    if f.read(2) == b'PK':
                        self.raw_data = pd.read_excel(self.file_path)
                    else:
                        try:
                            self.raw_data = pd.read_csv(self.file_path, encoding='utf-8')
                        except UnicodeDecodeError:
                            self.raw_data = pd.read_csv(self.file_path, encoding='latin-1')
            print(f"  > File Loaded. Raw Shape: {self.raw_data.shape}")
        except Exception as e:
            print(f"  > Error loading file: {e}")
            return None
        missing = [c for c in self.inputs if c not in self.raw_data.columns]
        if len(missing) > 5:
            print("  > DETECTED: Missing Composition Columns (Cast Dataset). Initiating text mining...")
            self.raw_data = self._perform_text_mining(self.raw_data)
        df = self.raw_data.copy()
        for std_name, aliases in self.target_map.items():
            for alias in aliases:
                if alias in df.columns:
                    df.rename(columns={alias: std_name}, inplace=True)
        cols = self.inputs + [t for t in self.targets if t in df.columns]
        for col in cols:
            df[col] = df[col].apply(self._clean_value)
        df[self.inputs] = df[self.inputs].fillna(0.0)
        avail = [t for t in self.targets if t in df.columns]
        df = df.dropna(subset=avail, how='all').reset_index(drop=True)
        self.clean_data = df
        print(f"  > PROCESSING COMPLETE. Training-Ready: {len(df)}")
        return self.clean_data

    def _perform_text_mining(self, df):
        def extract(text):
            if pd.isna(text): return {}
            text = str(text).replace('AI', 'Al')
            m = re.search(r'\\((.*?)\\\)', text)
            if not m: return {}
            formula = m.group(1)
            comp = {}
            for el in self.inputs:
                if el == 'Al': continue
                pat = re.compile(rf"{el}(\\d*\\.?\\d*)")
                hit = pat.search(formula)
                if hit:
                    val = hit.group(1)
                    comp[el] = float(val) if (val and val != '.') else 0.5
            total = sum(comp.values())
            comp['Al'] = 100 - total if 0 < total < 100 else 0.0
            return comp
        ext = df['Alloy Name'].apply(extract)
        comp_df = pd.DataFrame(list(ext)).fillna(0.0)
        for c in self.inputs:
            if c not in comp_df.columns: comp_df[c] = 0.0
        return pd.concat([df, comp_df], axis=1)

    def _clean_value(self, val):
        if pd.isna(val): return np.nan
        s = str(val).strip()
        if '-' in s:
            try: return sum([float(x) for x in re.findall(r'[0-9.]+', s)])/2
            except: pass
        m = re.search(r'[-+]?[0-9]*\.[0-9]+|[0-9]+', s)
        return float(m.group()) if m else np.nan

In [3]:
# MODULE 2: SUPER-ENSEMBLE (uses loaded hyperparams)
def build_estimator(name, params):
    p = params.copy() if params else get_default_hyperparams(name)
    if name == 'XGBoost':
        return xgb.XGBRegressor(objective='reg:squarederror', **{k: v for k, v in p.items()})
    if name == 'RandomForest':
        return RandomForestRegressor(**{k: v for k, v in p.items()})
    if name == 'GradientBoosting':
        return GradientBoostingRegressor(**{k: v for k, v in p.items()})
    return None

class EnsembleBrain:
    def __init__(self, data, inputs, targets, hyperparams=None):
        self.data = data
        self.inputs = inputs
        self.targets = [t for t in targets if t in data.columns]
        self.models = {}
        self.hp = hyperparams or load_hyperparams('wrought')

    def train_super_models(self):
        print(f"\n[{'='*25} PHASE 2: TRAINING SUPER-ENSEMBLE MODELS {'='*25}")
        for t in self.targets:
            df_t = self.data.dropna(subset=[t])
            if len(df_t) < 10: continue
            X = df_t[self.inputs]
            y = df_t[t]
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            est_xgb = build_estimator('XGBoost', self.hp.get('XGBoost'))
            est_rf = build_estimator('RandomForest', self.hp.get('RandomForest'))
            est_gb = build_estimator('GradientBoosting', self.hp.get('GradientBoosting'))
            super_model = VotingRegressor(estimators=[('xgb', est_xgb), ('rf', est_rf), ('gb', est_gb)])
            super_model.fit(X_train, y_train)
            preds = super_model.predict(X_test)
            r2 = r2_score(y_test, preds)
            mae = mean_absolute_error(y_test, preds)
            self.models[t] = super_model
            print(f"  > Super-Model for [{t[:15]:<15}]: R² = {r2*100:.1f}% | MAE = ±{mae:.2f}")

    def predict(self, df_input):
        results = {}
        for t, model in self.models.items():
            results[t] = model.predict(df_input)
        return pd.DataFrame(results, index=df_input.index)

In [4]:
# MODULE 3: GENERATIVE AI & BACKWARD (uses loaded GMM params)
class GenerativeDesignLab:
    def __init__(self, processor, brain, gmm_params=None):
        self.proc = processor
        self.brain = brain
        self.gen_model = None
        self.gmm_params = gmm_params or load_hyperparams('backward', 'GMM') or get_default_hyperparams('GMM')

    def train_generator(self):
        print(f"\n[{'='*25} PHASE 3: TRAINING GENERATIVE AI (GMM) {'='*25}")
        print("  > Learning the chemical 'DNA' of aluminum alloys...")
        X = self.proc.clean_data[self.proc.inputs]
        self.gen_model = GaussianMixture(**self.gmm_params)
        self.gen_model.fit(X)
        print("  > GenAI Training Complete.")

    def solve_backward_problem_multi_objective(self, target_uts, target_cond=None):
        print(f"\n[{'='*25} PHASE 4: BACKWARD MODELING {'='*25}")
        print(f"  > MISSION: Find alloy with UTS ≈ {target_uts} MPa", end='')
        if target_cond: print(f" AND Conductivity ≈ {target_cond} %IACS")
        else: print()
        n_samples = 50000
        samples, _ = self.gen_model.sample(n_samples)
        gen_df = pd.DataFrame(samples, columns=self.proc.inputs)
        gen_df[gen_df < 0] = 0
        gen_df['sum'] = gen_df.sum(axis=1)
        gen_df = gen_df[gen_df['sum'] > 0.1]
        for c in self.proc.inputs:
            gen_df[c] = (gen_df[c] / gen_df['sum']) * 100
        gen_df = gen_df.drop(columns=['sum']).fillna(0.0)
        print(f"  > Filtered to {len(gen_df)} valid chemical concepts.")
        preds = self.brain.predict(gen_df)
        full_df = pd.concat([gen_df.reset_index(drop=True), preds.reset_index(drop=True)], axis=1)
        full_df['Error_UTS'] = abs(full_df['UTS (MPa)'] - target_uts)
        if target_cond and 'Elec. Conductivity Vol (% IACS)' in full_df.columns:
            full_df['Error_Cond'] = abs(full_df['Elec. Conductivity Vol (% IACS)'] - target_cond)
            full_df['Total_Error'] = (full_df['Error_UTS']/500) + (full_df['Error_Cond']/50)
        else:
            full_df['Total_Error'] = full_df['Error_UTS']
        winners = full_df.sort_values('Total_Error').head(3)
        print(f"\n[{'='*25} OPTIMAL ALLOYS DISCOVERED {'='*25}]")
        for i, (idx, row) in enumerate(winners.iterrows()):
            print(f"\n--- CANDIDATE #{i+1} ---")
            print(f"  UTS: {row['UTS (MPa)']:.1f} MPa")
            if 'Elec. Conductivity Vol (% IACS)' in row:
                print(f"  Conductivity: {row['Elec. Conductivity Vol (% IACS)']:.1f} %IACS")
            recipe = [f"{el}={row[el]:.2f}%" for el in self.proc.inputs if row[el] > 0.1]
            print("  Recipe: " + ", ".join(recipe))
        if 'Elec. Conductivity Vol (% IACS)' in full_df.columns:
            plt.figure(figsize=(10, 6))
            plt.scatter(full_df['Elec. Conductivity Vol (% IACS)'], full_df['UTS (MPa)'], c='lightgray', s=5, alpha=0.5)
            plt.scatter(winners['Elec. Conductivity Vol (% IACS)'], winners['UTS (MPa)'], c='red', s=150, marker='*')
            plt.title(f"GenAI Pareto Optimization (Target: {target_uts} MPa)")
            plt.xlabel("Electrical Conductivity (% IACS)")
            plt.ylabel("UTS (MPa)")
            plt.legend(['Exploration', 'Optimal'])
            plt.grid(True, alpha=0.3)
            plt.show()

In [5]:
# MAIN EXECUTION
processor = TitaniumDataProcessor(FILE_NAME)
df_clean = processor.load_and_process()

if df_clean is not None:
    brain = EnsembleBrain(df_clean, processor.inputs, processor.targets)
    brain.train_super_models()
    lab = GenerativeDesignLab(processor, brain)
    lab.train_generator()
    lab.solve_backward_problem_multi_objective(target_uts=550, target_cond=40)

[========================= PHASE 1: DEEP DATA MINING =========================


  > File Loaded. Raw Shape: (868, 31)
  > PROCESSING COMPLETE. Training-Ready: 865

[========================= PHASE 2: TRAINING SUPER-ENSEMBLE MODELS =========================


  > Super-Model for [UTS (MPa)      ]: R² = 87.0% | MAE = ±35.51


  > Super-Model for [Yield Strength ]: R² = 64.2% | MAE = ±54.10

[========================= PHASE 3: TRAINING GENERATIVE AI (GMM) =========================
  > Learning the chemical 'DNA' of aluminum alloys...


  > GenAI Training Complete.

[========================= PHASE 4: BACKWARD MODELING =========================
  > MISSION: Find alloy with UTS ≈ 550 MPa AND Conductivity ≈ 40 %IACS
  > Filtered to 50000 valid chemical concepts.



[========================= OPTIMAL ALLOYS DISCOVERED =========================]

--- CANDIDATE #1 ---
  UTS: 549.9 MPa
  Recipe: Al=88.55%, Si=0.24%, Fe=0.30%, Cu=2.14%, Mn=0.19%, Mg=2.31%, Zn=6.06%, Ti=0.12%

--- CANDIDATE #2 ---
  UTS: 549.8 MPa
  Recipe: Al=88.52%, Si=0.25%, Fe=0.31%, Cu=2.15%, Mn=0.20%, Mg=2.30%, Zn=6.07%, Ti=0.12%

--- CANDIDATE #3 ---
  UTS: 549.8 MPa
  Recipe: Al=93.59%, Si=0.12%, Fe=0.15%, Cu=1.70%, Mn=0.10%, Mg=2.30%, Zn=2.00%
